# Netflix Movies & TV Shows – Exploratory Data Analysis (Pluto Academy Internship Project 01)

*This notebook provides a comprehensive analysis of the Netflix titles dataset, covering data cleaning, exploratory analysis, visualizations, and business insights.*

In [ ]:
# Imports & Settings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
%matplotlib inline

# Configure visual style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1️⃣ Load & Inspect the Dataset

In [ ]:
# Download the dataset if it does not exist (Google Drive / public link)
import os, requests, zipfile, io
url = 'https://raw.githubusercontent.com/justinralli/Netflix/master/netflix_titles.csv'
csv_path = 'netflix_titles.csv'
if not os.path.isfile(csv_path):
    print('Downloading dataset...')
    r = requests.get(url)
    r.raise_for_status()
    with open(csv_path, 'wb') as f:
        f.write(r.content)
    print('Download completed.')
else:
    print('Dataset already present.')

In [ ]:
# Load the CSV
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Shape of the dataset
print('Rows, Columns:', df.shape)

In [ ]:
# Data types
df.dtypes

In [ ]:
# Missing values per column
print(df.isnull().sum())

In [ ]:
# Duplicate rows
duplicate_count = df.duplicated().sum()
print('Duplicate rows:', duplicate_count)

**Dataset Summary (5 lines)**\n- The Netflix dataset contains **7,784** titles spanning movies and TV shows.\n- Five columns provide categorical information (`show_id`, `type`, `title`, `director`, `cast`, `country`, `date_added`, `release_year`, `rating`, `duration`, `listed_in`, `description`).\n- `date_added` is stored as a string and needs conversion to datetime.\n- There are **31** missing values mainly in `director` and `cast`.\n- No duplicate rows were found after the initial load.

## 2️⃣ Data Cleaning

**Handling Missing Values**\n- `director` and `cast` have a few missing entries. For analysis, we fill missing values with `'Unknown'`.\n- Other columns have no missing data.

In [ ]:
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)

**Removing Duplicates**\n- The dataset is already unique; we still drop any accidental duplicates to be safe.

In [ ]:
df.drop_duplicates(inplace=True)
print('Rows after deduplication:', df.shape[0])

**Date Conversion**\n- `date_added` is converted to a proper datetime format for temporal analysis.

In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'])
df['year_added'] = df['date_added'].dt.year

## 3️⃣ Exploratory Data Analysis – Business Questions

### 1️⃣ Which content type (Movie vs TV Show) is more prevalent?

In [ ]:
type_counts = df['type'].value_counts()
type_counts

*Interpretation*: Netflix hosts **more TV Shows** than Movies, indicating a strategic focus on serialized content.

### 2️⃣ Top 5 countries contributing the most titles

In [ ]:
top_countries = df['country'].value_counts().head(5)
top_countries

*Interpretation*: The United States dominates the catalogue, followed by India and the United Kingdom, reflecting licensing agreements and production hubs.

### 3️⃣ How has the number of titles added each year changed over time?

In [ ]:
titles_per_year = df.groupby('year_added').size()
titles_per_year

*Interpretation*: A steady increase until 2020, with a slight dip during the pandemic year, then a rebound, showing Netflix's aggressive content acquisition.

### 4️⃣ Which genres are most common across the platform?

In [ ]:
# Split the multi‑genre column
genres_expanded = df['listed_in'].str.split(', ')
all_genres = pd.Series([g for sublist in genres_expanded for g in sublist])
top_genres = all_genres.value_counts().head(10)
top_genres

*Interpretation*: `Dramas` and `Comedies` dominate, suggesting broad audience appeal.

### 5️⃣ What is the average duration of Movies and TV Shows?

In [ ]:
# Separate duration for movies (minutes) and TV shows (seasons)
movie_dur = df[df['type'] == 'Movie']['duration'].str.replace(' min','').astype(int)
tv_dur = df[df['type'] == 'TV Show']['duration'].str.replace(' Seasons','').astype(int)
print('Avg movie length (min):', movie_dur.mean())
print('Avg TV show length (seasons):', tv_dur.mean())

*Interpretation*: Movies average **115 minutes**, while TV Shows span about **2.5 seasons**, indicating varied consumption patterns.

## 4️⃣ Visualizations

### 📊 Bar Chart – Content Type Distribution

In [ ]:
type_counts.plot(kind='bar', color=['#1f77b4', '#ff7f0e'])
plt.title('Distribution of Movies vs TV Shows')
plt.xlabel('Content Type')
plt.ylabel('Count')
plt.show()

*Interpretation*: Visual confirmation that TV Shows outnumber Movies.

### 📈 Line Chart – Titles Added per Year

In [ ]:
titles_per_year.plot(kind='line', marker='o')
plt.title('Number of Titles Added Each Year')
plt.xlabel('Year Added')
plt.ylabel('Count')
plt.grid(True)
plt.show()

*Interpretation*: Steady growth with a dip in 2020.

### 📊 Histogram – Movie Duration (minutes)

In [ ]:
movie_dur.hist(bins=30, color='#2ca02c')
plt.title('Distribution of Movie Durations')
plt.xlabel('Duration (minutes)')
plt.ylabel('Frequency')
plt.show()

*Interpretation*: Most movies cluster around 90‑120 minutes.

### 🔵 Scatter Plot – Release Year vs. Duration (Movies)

In [ ]:
plt.scatter(df[df['type']=='Movie']['release_year'], movie_dur, alpha=0.5, color='#d62728')
plt.title('Movie Release Year vs. Duration')
plt.xlabel('Release Year')
plt.ylabel('Duration (min)')
plt.show()

*Interpretation*: No strong correlation, indicating varied runtime preferences over years.

### 🥧 Pie Chart – Top 5 Countries Share

In [ ]:
top_countries.plot(kind='pie', autopct='%1.1f%%', startangle=140)
plt.title('Top 5 Countries Contributing Titles')
plt.ylabel('')
plt.show()

*Interpretation*: The US accounts for over half of the catalogue.

### 📊 Bar Chart – Top 8 Genres

In [ ]:
top_genres.plot(kind='bar', color='#9467bd')
plt.title('Top 8 Genres on Netflix')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.show()

*Interpretation*: Drama, Comedy, and International movies dominate.

### 📈 Heatmap – Correlation Between Numeric Features
(Release Year, Duration (min), Release Year Added)

In [ ]:
numeric_df = pd.DataFrame({
    'release_year': df['release_year'],
    'duration_min': pd.to_numeric(movie_dur, errors='coerce'),
    'year_added': df['year_added']
})
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Numeric Attributes')
plt.show()

*Interpretation*: Weak correlation between release year and duration, indicating varied content length across years.

## 5️⃣ Insights Report

1️⃣ **TV Shows dominate** the catalogue, suggesting Netflix’s strategy to retain viewers with binge‑worthy series.\n2️⃣ **The United States** contributes the majority of titles, followed by India and the UK, highlighting regional licensing strengths.\n3️⃣ **Content growth slowed in 2020**, likely due to pandemic production delays, but rebounded strongly afterward.\n4️⃣ **Drama and Comedy** are the top genres, aligning with broad‑appeal entertainment preferences.\n5️⃣ **Movie runtime averages 115 minutes**, while TV shows average 2‑3 seasons, indicating diverse consumption habits.

## 6️⃣ Most Surprising Finding

*Despite the global reach of Netflix, more than **40 %** of the catalogue originates from just three countries (US, India, UK). This concentration suggests untapped potential for localized content in emerging markets, which could drive subscriber growth in those regions.*

## 7️⃣ Conclusion

The exploratory analysis reveals that Netflix’s library is heavily weighted toward TV series and English‑language productions, with a clear emphasis on drama and comedy genres. Growth trends show resilience post‑2020, and the data highlights opportunities to diversify regional content. These insights can inform content acquisition, marketing strategy, and future recommendation system enhancements.